# Tarea 3 — PyTorch

**Nombre del integrante:** Christian Goncalves

**Cédula del integrante:** 30224303

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import copy
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms


SEED = 42
BATCH_SIZE = 128
TASK_DIR = Path.cwd()
if (TASK_DIR / "tarea-3-codigo" / "README.md").is_file():
    TASK_DIR = TASK_DIR / "tarea-3-codigo"
DATA_DIR = TASK_DIR / "data"
NOTEBOOK_PREFIX = "pytorch_fashion_2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ORIGINAL_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
SIMPLIFIED_CLASSES = ["Top", "Bottom", "Footwear", "Bag"]
SIMPLIFIED_LABELS = (0, 1, 0, 1, 0, 2, 0, 2, 3, 2)
CLASS_NAMES = {"original": ORIGINAL_CLASSES, "simplified": SIMPLIFIED_CLASSES}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")

## 2. Datos (Fashion MNIST original y clases simplificadas)

In [ ]:
class FashionMNISTView(Dataset):
    def __init__(self, dataset, classification):
        self.dataset = dataset
        self.classification = classification

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, label = self.dataset[index]
        if self.classification == "simplified":
            label = SIMPLIFIED_LABELS[label]
        return image, label


transform = transforms.ToTensor()
base_train_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=True, download=True, transform=transform
)
base_test_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=False, download=True, transform=transform
)


def get_dataloaders(classification, use_validation=False):
    if classification not in CLASS_NAMES:
        raise ValueError(f"Unsupported classification: {classification}")

    train_dataset = FashionMNISTView(base_train_dataset, classification)
    test_dataset = FashionMNISTView(base_test_dataset, classification)
    if use_validation:
        validation_size = int(0.1 * len(train_dataset))
        training_size = len(train_dataset) - validation_size
        train_dataset, validation_dataset = random_split(
            train_dataset,
            [training_size, validation_size],
            generator=torch.Generator().manual_seed(SEED),
        )
        validation_loader = DataLoader(
            validation_dataset, batch_size=BATCH_SIZE, shuffle=False
        )
    else:
        validation_loader = None

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True
    )
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, validation_loader, test_loader


print(f"Training examples: {len(base_train_dataset)}")
print(f"Test examples: {len(base_test_dataset)}")

## 3. Modelos (rectangular y convolucional)

In [ ]:
def calculate_hidden_neurons(network_type, hidden_layers, neurons_per_layer):
    if hidden_layers < 0:
        raise ValueError("The number of hidden layers cannot be negative.")
    if network_type == "rectangular":
        if neurons_per_layer < 1:
            raise ValueError("Each hidden layer must contain at least one neuron.")
        return [neurons_per_layer] * hidden_layers
    if network_type == "convolutional":
        return [
            round(784 + (10 - 784) * index / (hidden_layers + 1))
            for index in range(1, hidden_layers + 1)
        ]
    raise ValueError(f"Unsupported network type: {network_type}")


def create_model(network_type, hidden_neurons, output_classes, optimizations):
    if network_type not in ("rectangular", "convolutional"):
        raise ValueError(f"Unsupported network type: {network_type}")
    layers = [nn.Flatten()]
    input_neurons = 784
    for neuron_count in hidden_neurons:
        layers.append(nn.Linear(input_neurons, neuron_count))
        if optimizations["batch_norm"]:
            layers.append(nn.BatchNorm1d(neuron_count))
        layers.append(nn.ReLU())
        if optimizations["dropout"]:
            layers.append(nn.Dropout(p=optimizations["dropout_rate"]))
        input_neurons = neuron_count
    layers.append(nn.Linear(input_neurons, output_classes))
    return nn.Sequential(*layers).to(DEVICE)


model = None
hyperparameters = None
print("Model definitions are ready.")

## 4. Entrenar, probar, guardar y cargar

In [ ]:
def ask_integer(prompt, default, minimum):
    while True:
        response = input(f"{prompt} [{default}]: ").strip()
        if not response:
            return default
        try:
            value = int(response)
        except ValueError:
            print("Ingresa un numero entero.")
            continue
        if value < minimum:
            print(f"Ingresa un valor mayor o igual a {minimum}.")
            continue
        return value


def ask_float(prompt, default, minimum, maximum=None):
    while True:
        response = input(f"{prompt} [{default}]: ").strip()
        if not response:
            return default
        try:
            value = float(response)
        except ValueError:
            print("Ingresa un valor numerico.")
            continue
        if (
            not math.isfinite(value) or value < minimum
            or (maximum is not None and value > maximum)
        ):
            print(f"Ingresa un valor entre {minimum} y {maximum or 'infinito'}.")
            continue
        return value


def ask_yes_no(prompt, default=False):
    default_text = "S/n" if default else "s/N"
    while True:
        response = input(f"{prompt} ({default_text}): ").strip().lower()
        if not response:
            return default
        if response in ("s", "si", "y", "yes"):
            return True
        if response in ("n", "no"):
            return False
        print("Responde si o no.")


def choose_option(prompt, options, default, display_options=None):
    if display_options is None:
        display_options = options
    while True:
        print(prompt)
        for number, label in enumerate(display_options, start=1):
            print(f"  {number}. {label}")
        response = input(f"Opcion [{default}]: ").strip()
        if not response:
            return options[default - 1]
        try:
            selected = int(response)
        except ValueError:
            print("Selecciona un numero de la lista.")
            continue
        if 1 <= selected <= len(options):
            return options[selected - 1]
        print("La opcion no existe.")


def create_new_model():
    global model, hyperparameters

    classification = choose_option(
        "Selecciona la clasificacion:", ["original", "simplified"], 1,
        ["Fashion MNIST original (10 clases)", "Clases simplificadas (4 clases)"],
    )
    network_type = choose_option(
        "Selecciona la arquitectura:", ["rectangular", "convolutional"], 1,
        ["Rectangular", "Convolucional"],
    )
    hidden_layers = ask_integer("Numero de capas ocultas", 2, 0)
    neurons_per_layer = 512
    if network_type == "rectangular" and hidden_layers:
        neurons_per_layer = ask_integer("Neuronas por capa oculta", 512, 1)
    hidden_neurons = calculate_hidden_neurons(
        network_type, hidden_layers, neurons_per_layer
    )

    use_batch_norm = ask_yes_no("Habilitar Batch Normalization?")
    use_dropout = ask_yes_no("Habilitar Dropout?")
    dropout_rate = ask_float("Tasa de Dropout", 0.25, 0.0, 0.99) if use_dropout else 0.0
    use_weight_decay = ask_yes_no("Habilitar weight decay?")
    weight_decay_value = (
        ask_float("Valor de weight decay", 0.0001, 0.0)
        if use_weight_decay else 0.0
    )
    use_early_stopping = ask_yes_no("Habilitar early stopping?")
    patience = ask_integer("Paciencia de early stopping", 3, 1) if use_early_stopping else 3
    optimizer_name = choose_option(
        "Selecciona el optimizador:", ["Adam", "SGD"], 1
    )
    learning_rate = ask_float("Learning rate", 0.001, 0.0000001)

    optimizations = {
        "batch_norm": use_batch_norm,
        "dropout": use_dropout,
        "dropout_rate": dropout_rate,
        "weight_decay": use_weight_decay,
        "weight_decay_value": weight_decay_value,
        "early_stopping": use_early_stopping,
        "patience": patience,
    }
    hyperparameters = {
        "network_type": network_type,
        "hidden_layers": hidden_layers,
        "neurons_per_layer": hidden_neurons,
        "epochs": 0,
        "classification": classification,
        "optimizations": optimizations,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "batch_size": BATCH_SIZE,
    }
    model = create_model(
        network_type,
        hidden_neurons,
        len(CLASS_NAMES[classification]),
        optimizations,
    )
    print(f"Created {network_type} model for {classification} classification.")
    if network_type == "convolutional":
        print(f"Interpolated hidden-layer sizes: {hidden_neurons}")


def evaluate_loss(data_loader):
    criterion = nn.CrossEntropyLoss()
    model.eval()
    total_loss = 0.0
    total_examples = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            loss = criterion(model(images), labels)
            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)
    return total_loss / total_examples


def train_model(epochs):
    global hyperparameters
    if model is None or hyperparameters is None:
        print("Primero debes crear o cargar una red.")
        return
    if epochs < 1:
        raise ValueError("The number of epochs must be at least one.")

    use_early_stopping = hyperparameters["optimizations"]["early_stopping"]
    train_loader, validation_loader, _ = get_dataloaders(
        hyperparameters["classification"], use_validation=use_early_stopping
    )
    optimizer_class = optim.Adam if hyperparameters["optimizer"] == "Adam" else optim.SGD
    optimizer_options = {
        "lr": hyperparameters["learning_rate"],
        "weight_decay": hyperparameters["optimizations"]["weight_decay_value"],
    }
    if hyperparameters["optimizer"] == "SGD":
        optimizer_options["momentum"] = 0.9
    optimizer = optimizer_class(model.parameters(), **optimizer_options)
    criterion = nn.CrossEntropyLoss()
    losses = []
    best_state = None
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    completed_epochs = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)
        epoch_loss = total_loss / total_examples
        losses.append(epoch_loss)
        completed_epochs += 1
        print(f"Epoch {epoch + 1}/{epochs} - loss: {epoch_loss:.4f}")

        if validation_loader is not None:
            validation_loss = evaluate_loss(validation_loader)
            print(f"Validation loss: {validation_loss:.4f}")
            if validation_loss < best_validation_loss:
                best_validation_loss = validation_loss
                best_state = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= hyperparameters["optimizations"]["patience"]:
                    print("Early stopping: validation loss did not improve.")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    hyperparameters["epochs"] += completed_epochs

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker="o")
    plt.title("Training loss by epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def test_model():
    if model is None or hyperparameters is None:
        print("Primero debes crear o cargar una red.")
        return

    _, _, test_loader = get_dataloaders(hyperparameters["classification"])
    model.eval()
    actual_labels = []
    predicted_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = model(images.to(DEVICE)).argmax(dim=1).cpu()
            actual_labels.extend(labels.numpy().tolist())
            predicted_labels.extend(predictions.numpy().tolist())

    class_count = len(CLASS_NAMES[hyperparameters["classification"]])
    confusion = np.zeros((class_count, class_count), dtype=np.int64)
    np.add.at(confusion, (actual_labels, predicted_labels), 1)
    true_positives = np.diag(confusion).astype(float)
    predicted_counts = confusion.sum(axis=0)
    actual_counts = confusion.sum(axis=1)
    precision_by_class = np.divide(
        true_positives, predicted_counts,
        out=np.zeros_like(true_positives), where=predicted_counts != 0,
    )
    recall_by_class = np.divide(
        true_positives, actual_counts,
        out=np.zeros_like(true_positives), where=actual_counts != 0,
    )
    accuracy = float(true_positives.sum() / confusion.sum())
    names = CLASS_NAMES[hyperparameters["classification"]]

    print("Matriz de confusion (filas: clase real; columnas: prediccion):")
    print(confusion)
    print("\nPrecision y recall por clase:")
    for index, name in enumerate(names):
        print(
            f"{name}: precision={precision_by_class[index]:.4f}, "
            f"recall={recall_by_class[index]:.4f}"
        )
    print(f"Macro precision: {precision_by_class.mean():.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro recall: {recall_by_class.mean():.4f}")

    figure, axis = plt.subplots(figsize=(8, 6))
    image = axis.imshow(confusion, cmap="Blues")
    figure.colorbar(image, ax=axis)
    axis.set(
        xticks=np.arange(class_count), yticks=np.arange(class_count),
        xticklabels=names, yticklabels=names,
        xlabel="Prediccion", ylabel="Clase real", title="Matriz de confusion",
    )
    plt.setp(axis.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    threshold = confusion.max() / 2
    for row in range(class_count):
        for column in range(class_count):
            axis.text(
                column, row, str(confusion[row, column]),
                ha="center", va="center",
                color="white" if confusion[row, column] > threshold else "black",
            )
    figure.tight_layout()
    plt.show()


def default_checkpoint_name():
    return (
        f"{NOTEBOOK_PREFIX}_{hyperparameters['classification']}_"
        f"{hyperparameters['network_type']}.pth"
    )


def save_model():
    if model is None or hyperparameters is None:
        print("Primero debes crear o cargar una red.")
        return
    default_name = default_checkpoint_name()
    response = input(f"Ruta del archivo [{default_name}]: ").strip()
    checkpoint_path = Path(response) if response else TASK_DIR / default_name
    if checkpoint_path.suffix.lower() != ".pth":
        print("El archivo de la red debe tener extension .pth.")
        return
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "hyperparameters": copy.deepcopy(hyperparameters),
        },
        checkpoint_path,
    )
    print(f"Model saved to {checkpoint_path}.")


def load_model():
    global model, hyperparameters
    checkpoint_path = Path(input("Ruta del archivo .pth: ").strip())
    if not checkpoint_path.is_file():
        print(f"No se encontro el archivo: {checkpoint_path}")
        return
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    if not isinstance(checkpoint, dict) or "state_dict" not in checkpoint:
        raise ValueError("The checkpoint must contain a state_dict.")
    loaded_hyperparameters = checkpoint.get("hyperparameters")
    required_keys = {
        "network_type", "hidden_layers", "neurons_per_layer", "epochs",
        "classification", "optimizations", "optimizer", "learning_rate",
    }
    if not isinstance(loaded_hyperparameters, dict) or not required_keys.issubset(loaded_hyperparameters):
        raise ValueError("The checkpoint is missing required hyperparameters.")
    classification = loaded_hyperparameters["classification"]
    if classification not in CLASS_NAMES:
        raise ValueError(f"Unsupported classification in checkpoint: {classification}")
    model = create_model(
        loaded_hyperparameters["network_type"],
        loaded_hyperparameters["neurons_per_layer"],
        len(CLASS_NAMES[classification]),
        loaded_hyperparameters["optimizations"],
    )
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    hyperparameters = loaded_hyperparameters
    print(f"Model loaded from {checkpoint_path}.")


def run_menu():
    actions = {
        "1": create_new_model,
        "2": load_model,
        "3": lambda: train_model(ask_integer("Numero de epocas", 10, 1)),
        "4": test_model,
        "5": save_model,
    }
    while True:
        print("\n=== Clasificador Fashion MNIST ===")
        print("1. Crear una red nueva")
        print("2. Cargar una red desde un archivo .pth")
        print("3. Entrenar la red actual")
        print("4. Probar la red actual")
        print("5. Guardar la red actual")
        print("6. Salir")
        choice = input("Selecciona una opcion: ").strip()
        if choice == "6":
            print("Programa finalizado.")
            break
        action = actions.get(choice)
        if action is None:
            print("Opcion no valida. Intenta de nuevo.")
            continue
        action()

## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
run_menu()